# Assignment 1 — Gradio dashboard: French bakery daily revenue

Loads **only** what `train.ipynb` saved in `artifacts/` (models, scaler,
predictions, features). Nothing is trained here.

In [ ]:
import json
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch

ARTIFACTS = Path("artifacts")
meta = json.loads((ARTIFACTS / "meta.json").read_text())
FEATURES = meta["features"]
Y_MEAN, Y_STD = meta["y_mean"], meta["y_std"]

preds = pd.read_csv(ARTIFACTS / "predictions.csv", parse_dates=["date"])
feat = pd.read_csv(ARTIFACTS / "features.csv", parse_dates=["date"])
closed_est = pd.read_csv(ARTIFACTS / "closed_days_estimate.csv", parse_dates=["date"])

MODELS = ["sklearn", "torch_manual", "torch_module"]
BASELINES = ["naive_mean", "seasonal_naive"]
LABELS = {
    "sklearn": "scikit-learn",
    "torch_manual": "PyTorch manual loop",
    "torch_module": "PyTorch nn.Module",
    "naive_mean": "Baseline: train mean",
    "seasonal_naive": "Baseline: same weekday last week",
}

## Load the saved models (no retraining)

In [ ]:
scaler = joblib.load(ARTIFACTS / "scaler.joblib")
sk_model = joblib.load(ARTIFACTS / "sklearn_ridge.joblib")

manual = torch.load(ARTIFACTS / "torch_manual.pt")
w_manual, b_manual = manual["w"], manual["b"]


class LinearRegressor(torch.nn.Module):  # same architecture as in train.ipynb
    def __init__(self, n_in):
        super().__init__()
        self.linear = torch.nn.Linear(n_in, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)


torch_model = LinearRegressor(len(FEATURES))
torch_model.load_state_dict(torch.load(ARTIFACTS / "torch_module.pt"))
torch_model.eval()


def predict_all(X_raw):
    # X_raw: DataFrame with FEATURES columns -> EUR predictions of the three models.
    Xs = scaler.transform(X_raw[FEATURES]).astype(np.float32)
    Xt = torch.tensor(Xs)
    with torch.no_grad():
        return {
            "sklearn": sk_model.predict(Xs),
            "torch_manual": (Xt @ w_manual + b_manual).numpy() * Y_STD + Y_MEAN,
            "torch_module": torch_model(Xt).numpy() * Y_STD + Y_MEAN,
        }


# Sanity check: reloaded models reproduce the saved test predictions.
test = preds[preds["split"] == "test"]
check = predict_all(feat.set_index("date").loc[test["date"]])
for m in MODELS:
    assert np.allclose(check[m], test[m], atol=0.05), m
print("reloaded models reproduce saved predictions")

## Views

In [ ]:
SPLIT_COLORS = {"train": "rgba(150,150,150,0.08)", "val": "rgba(255,165,0,0.12)", "test": "rgba(0,128,255,0.10)"}


def metrics_table(split):
    d = preds[preds["split"] == split]
    rows = []
    for m in BASELINES + MODELS:
        err = d[m] - d["actual"]
        rows.append({
            "model": LABELS[m],
            "MAE (EUR/day)": round(err.abs().mean(), 2),
            "RMSE": round((err ** 2).mean() ** 0.5, 2),
            "R2": round(1 - (err ** 2).sum() / ((d["actual"] - d["actual"].mean()) ** 2).sum(), 3),
            "% days under-forecast": round(100 * (err < 0).mean(), 1),
        })
    return pd.DataFrame(rows)


def timeseries_plot(models, split):
    d = preds if split == "all" else preds[preds["split"] == split]
    fig = go.Figure()
    if split == "all":
        for s, (lo, hi) in d.groupby("split")["date"].agg(["min", "max"]).iterrows():
            fig.add_vrect(x0=lo, x1=hi, fillcolor=SPLIT_COLORS[s], line_width=0, annotation_text=s, annotation_position="top left")
    fig.add_trace(go.Scatter(x=d["date"], y=d["actual"], name="actual", line=dict(color="black", width=2)))
    for m in models:
        fig.add_trace(go.Scatter(x=d["date"], y=d[m], name=LABELS[m], line=dict(width=1.2)))
    fig.update_layout(title="Daily revenue: prediction vs actual (open days)", yaxis_title="EUR / day", height=430, legend=dict(orientation="h", y=-0.2))
    return fig


def scatter_plot(split):
    d = preds[preds["split"] == split]
    long = d.melt(id_vars=["date", "actual"], value_vars=MODELS, var_name="model", value_name="predicted")
    long["model"] = long["model"].map(LABELS)
    fig = px.scatter(long, x="actual", y="predicted", color="model", opacity=0.6, facet_col="model", height=380,
                     title=f"Predicted vs actual — {split} set (dashed line = perfect prediction)")
    lo, hi = d["actual"].min(), d["actual"].max()
    for i in range(1, 4):
        fig.add_shape(type="line", x0=lo, y0=lo, x1=hi, y1=hi, line=dict(dash="dash", color="grey"), row=1, col=i)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.update_layout(showlegend=False)
    return fig


def agreement_text():
    d = preds
    diffs = {m: (d[m] - d["sklearn"]).abs().max() for m in ["torch_manual", "torch_module"]}
    return (f"**Do the three implementations agree?** Largest difference from scikit-learn on any day: "
            f"manual loop {diffs['torch_manual']:.3f} EUR, nn.Module {diffs['torch_module']:.3f} EUR.")

In [ ]:
open_feat = feat[~feat["closed"]].copy()
open_feat["weekday"] = open_feat["date"].dt.day_name()
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
NUMERIC_FEATURES = ["revenue", "lag7", "roll4", "temp_max_c", "precip_mm", "sunshine_h"]
BINARY_FEATURES = ["school_holiday", "public_holiday", "weekend_summer"]


def target_plots():
    h = px.histogram(open_feat, x="revenue", nbins=50, title="Distribution of daily revenue (open days)", height=330)
    h.update_layout(xaxis_title="EUR / day")
    b = px.box(open_feat, x="weekday", y="revenue", category_orders={"weekday": WEEKDAYS}, title="Revenue by weekday", height=330)
    m = open_feat.groupby(open_feat["date"].dt.month)["revenue"].mean().reset_index()
    mb = px.bar(m, x="date", y="revenue", title="Mean revenue by month", height=330, labels={"date": "month", "revenue": "EUR / day"})
    return h, b, mb


def feature_plot(name):
    if name in BINARY_FEATURES:
        d = open_feat.groupby(name)["revenue"].agg(["mean", "count"]).reset_index()
        d[name] = d[name].map({0.0: "no", 1.0: "yes"})
        return px.bar(d, x=name, y="mean", text="count", title=f"Mean revenue by {name} (bar label = number of days)",
                      labels={"mean": "EUR / day"}, height=380)
    fig = px.scatter(open_feat, x=name, y="revenue", opacity=0.5, marginal_x="histogram",
                     title=f"{name}: distribution and relation with revenue", height=420)
    if name in ["temp_max_c", "precip_mm", "sunshine_h"]:
        fig.add_annotation(text="weather was tested but NOT used in the final model", xref="paper", yref="paper", x=0, y=1.08, showarrow=False)
    return fig

In [ ]:
def business_cost(under_cost, over_cost, margin_pct):
    # Cost of forecast errors on the test set when the rota follows the forecast (+ an optional safety margin).
    d = preds[preds["split"] == "test"]
    rows = []
    for m in BASELINES + MODELS:
        planned = d[m] * (1 + margin_pct / 100)
        under = (d["actual"] - planned).clip(lower=0)   # demand above plan: lost sales / queues
        over = (planned - d["actual"]).clip(lower=0)    # plan above demand: idle staff
        rows.append({
            "model": LABELS[m],
            "EUR under-forecast": round(under.sum()),
            "EUR over-forecast": round(over.sum()),
            "total cost (EUR)": round(under_cost * under.sum() + over_cost * over.sum()),
        })
    table = pd.DataFrame(rows)

    # The margin is a decision, so it is chosen on validation; test only shows what that choice would have cost.
    margins = np.arange(-20, 41, 2)

    def cost_curve(split):
        dd = preds[preds["split"] == split]
        planned = dd["sklearn"].to_numpy()[:, None] * (1 + margins / 100)
        actual = dd["actual"].to_numpy()[:, None]
        return under_cost * (actual - planned).clip(min=0).sum(0) + over_cost * (planned - actual).clip(min=0).sum(0)

    val_curve, test_curve = cost_curve("val"), cost_curve("test")
    best = margins[val_curve.argmin()]
    curves = pd.DataFrame({"margin": margins, "validation (Apr-May, chooses the margin)": val_curve,
                           "test (Jun-Sep, checks it)": test_curve}).melt("margin", var_name="period", value_name="cost")
    fig = px.line(curves, x="margin", y="cost", color="period",
                  labels={"margin": "safety margin added to the forecast (%)", "cost": "total cost (EUR)"},
                  title="scikit-learn model: cost vs safety margin", height=350)
    fig.add_vline(x=margin_pct, line_dash="dash")
    fig.add_vline(x=best, line_dash="dot", line_color="green")
    no_margin, chosen = test_curve[margins == 0][0], test_curve[margins == best][0]
    note = (f"With these costs, validation picks a margin of **{best:+d}%** (green dotted line). "
            f"Applied to the test period it costs **{chosen:,.0f} EUR** instead of {no_margin:,.0f} EUR with no margin "
            f"({1 - chosen / no_margin:.0%} less). The cheapest margin *in hindsight* on test would have been "
            f"{margins[test_curve.argmin()]:+d}%, but that uses the test set to decide, so it is shown only for reference.")
    return table, fig, note


def closed_days_view():
    d = closed_est.copy()
    d["weekday"] = d["date"].dt.day_name()
    d["estimated_revenue"] = d["estimated_revenue"].round()
    fig = px.bar(d, x="date", y="estimated_revenue", color="weekday", title="Estimated revenue lost on days the bakery was closed", height=360,
                 labels={"estimated_revenue": "EUR"})
    total = f"**{len(d)} closed days** could be estimated → about **{d['estimated_revenue'].sum():,.0f} EUR** of lost revenue."
    return fig, d[["date", "weekday", "estimated_revenue"]].assign(date=d["date"].dt.date), total

In [ ]:
school = pd.read_csv("data/school_holidays_zone_b.csv", parse_dates=["start_date", "end_date"])
public = set(pd.to_datetime(pd.read_csv("data/public_holidays_fr.csv")["date"]).dt.date)


def what_if(date_str, lag7, roll4, school_holiday, public_holiday):
    try:
        day = pd.Timestamp(date_str)
    except ValueError:
        return "Enter a date as YYYY-MM-DD.", None
    row = {f: 0.0 for f in FEATURES}
    if day.dayofweek > 0:
        row[f"dow_{day.dayofweek}"] = 1.0
    if day.month > 1:
        row[f"month_{day.month}"] = 1.0
    row.update(lag7=lag7, roll4=roll4, school_holiday=float(school_holiday), public_holiday=float(public_holiday),
               weekend_summer=float(day.dayofweek >= 5 and day.month in (7, 8)))
    p = predict_all(pd.DataFrame([row]))
    table = pd.DataFrame({"model": [LABELS[m] for m in MODELS], "forecast (EUR)": [round(float(p[m][0]), 1) for m in MODELS]})
    return f"Forecast for **{day.day_name()} {day.date()}**", table


def default_flags(date_str):
    try:
        day = pd.Timestamp(date_str)
    except ValueError:
        return gr.update(), gr.update()
    in_school = bool(((school["start_date"] <= day) & (day < school["end_date"])).any())
    return in_school, day.date() in public

## Layout

In [ ]:
with gr.Blocks(title="Bakery revenue forecast") as demo:
    gr.Markdown(
        "# 🥐 French bakery — daily revenue forecast\n"
        "Forecast made **7 days ahead** so the owner can plan the staff rota. "
        "Same model (ridge linear regression) trained three ways. All views use the saved models and predictions."
    )

    with gr.Tab("Prediction vs actual"):
        split_dd = gr.Radio(["test", "val", "train", "all"], value="test", label="Period")
        model_cb = gr.CheckboxGroup(MODELS + BASELINES, value=MODELS, label="Series to plot")
        ts = gr.Plot()
        sc = gr.Plot()
        gr.Markdown(agreement_text())
        mt = gr.Dataframe(label="Metrics on the selected period")

        def refresh(split, models):
            s = "test" if split == "all" else split
            return timeseries_plot(models, split), scatter_plot(s), metrics_table(s)

        for c in (split_dd, model_cb):
            c.change(refresh, [split_dd, model_cb], [ts, sc, mt])
        demo.load(refresh, [split_dd, model_cb], [ts, sc, mt])

    with gr.Tab("Distributions"):
        with gr.Row():
            h1, h2, h3 = gr.Plot(), gr.Plot(), gr.Plot()
        feat_dd = gr.Dropdown(NUMERIC_FEATURES[1:] + BINARY_FEATURES, value="roll4", label="Feature")
        fp = gr.Plot()
        feat_dd.change(feature_plot, feat_dd, fp)
        demo.load(target_plots, None, [h1, h2, h3])
        demo.load(feature_plot, feat_dd, fp)

    with gr.Tab("Business cost"):
        gr.Markdown(
            "Cost of forecast errors on the **test period**. Costs per EUR of error are *assumptions* "
            "(see REPORT.md): being short-staffed loses sales, being over-staffed pays idle hours."
        )
        with gr.Row():
            u = gr.Slider(0, 1, value=0.6, step=0.05, label="Cost per EUR under-forecast (lost margin)")
            o = gr.Slider(0, 1, value=0.3, step=0.05, label="Cost per EUR over-forecast (idle staff)")
            mg = gr.Slider(-20, 40, value=0, step=1, label="Safety margin added to forecast (%)")
        note = gr.Markdown()
        ct = gr.Dataframe()
        cf = gr.Plot()
        for c in (u, o, mg):
            c.change(business_cost, [u, o, mg], [ct, cf, note])
        demo.load(business_cost, [u, o, mg], [ct, cf, note])

    with gr.Tab("Closed days"):
        gr.Markdown("Revenue the model expects the bakery would have made on the days it was closed.")
        total_md = gr.Markdown()
        cdf = gr.Plot()
        cdt = gr.Dataframe()
        demo.load(closed_days_view, None, [cdf, cdt, total_md])

    with gr.Tab("What-if forecast"):
        gr.Markdown("Build a day and get the forecast from the three saved models.")
        with gr.Row():
            date_in = gr.Textbox(value="2022-08-15", label="Date (YYYY-MM-DD)")
            lag_in = gr.Number(value=1800, label="Revenue same weekday last week (EUR)")
            roll_in = gr.Number(value=1700, label="Mean of last 4 same weekdays (EUR)")
        with gr.Row():
            sch_in = gr.Checkbox(value=True, label="School holiday (Zone B)")
            pub_in = gr.Checkbox(value=True, label="Public holiday")
        btn = gr.Button("Forecast", variant="primary")
        wi_md = gr.Markdown()
        wi_t = gr.Dataframe()
        date_in.change(default_flags, date_in, [sch_in, pub_in])
        btn.click(what_if, [date_in, lag_in, roll_in, sch_in, pub_in], [wi_md, wi_t])

# inline=False: inside Jupyter / VS Code the embedded preview often renders blank,
# so always open the dashboard in a real browser tab (http://127.0.0.1:7860).
# close_all() frees the port if this cell was already run in the same kernel.
gr.close_all()
demo.launch(server_name="127.0.0.1", server_port=7860, inbrowser=True, inline=False)